# Few-shot candidate selection 
**— from train_pool, without touching eval.**

- Zero-shot error analysis: most errors are in the neutral → negative direction.
- The model seems to be rejecting complaints based on the reason given. 
Examples should demonstrate this boundary—and include examples from BOTH sides of the boundary; otherwise, 
the model will learn to infer “I saw a complaint → neutral.”

In [8]:
import pandas as pd

from src.config import DATA_PROCESSED, RESULTS, SEED, LABELS

N_PER_CLASS = 8
MIN_LEN, MAX_LEN = 200, 450
CONTRAST = r"\b(?:but|however|though|although|wish|expecting|hoping|overall)\b"

train = pd.read_csv(DATA_PROCESSED / "train_pool.csv")
train["n_chars"] = train["text"].str.len()

hit = train["text"].str.contains(CONTRAST, case=False, regex=True)
cand = train[hit & train["n_chars"].between(MIN_LEN, MAX_LEN)]


In [9]:
lines = [
    "# Few-shot candidates",
    "",
    f"Source: `train_pool.csv` (n={len(train):,}) — the same pool the baseline",
    "trains on, so both methods draw from identical data. `eval` is untouched.",
    "",
    f"Filter: contrast marker present, length {MIN_LEN}–{MAX_LEN} chars.",
    f"Candidate pool: **{len(cand):,}**",
    "",
    "| class | candidates | in pool | rate |",
    "|---|---|---|---|",
]
for lbl in LABELS:
    n_c = int((cand["label"] == lbl).sum())
    n_p = int((train["label"] == lbl).sum())
    lines.append(f"| {lbl} | {n_c:,} | {n_p:,} | {n_c / n_p:.0%} |")

lines += ["", "Contrast-marker rate is highest for neutral — consistent with the EDA",
          "finding, though the margin is modest.", ""]

for lbl in LABELS:
    sub = cand[cand["label"] == lbl].sample(N_PER_CLASS, random_state=SEED)
    lines += [f"## {lbl.upper()}", ""]
    for idx, r in sub.iterrows():
        lines += [f"**[{idx}]** ({r['n_chars']} chars)", "", f"> {r['text']}", ""]


In [10]:
out = RESULTS / "fewshot_candidates.md"
out.write_text("\n".join(lines), encoding="utf-8")

print(f"{len(cand):,} aday → {out}")
print(cand["label"].value_counts().reindex(LABELS).to_string())

5,897 aday → /workspaces/llm-vs-classic-ml-text-classification/results/fewshot_candidates.md
label
negative     996
neutral      663
positive    4238


- NEUTRAL: overall ... okay ,  11944, 23432
- NEGATIVE:  5457
- POSITIVE:  22890